# Tuned Support Vector Regression Model
## T2 Immunological Data
### Target: pain_under_load_reduction_pct

In [ ]:
import sys, os
sys.path.insert(0, '../src')
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from skrub import TableReport
import joblib, os
import preprocess
import explore
import model
import model_svr
from sklearn.preprocessing import PowerTransformer
os.environ['PYTHONWARNINGS'] = 'ignore'

# Path to save results
MODEL_DIR = os.path.abspath('../models')

In [ ]:
# Running through preprocessing steps
# Load raw data
df_im, df_cl = explore.load_data()

# Clean datasets
_im_id_cols = ['Patient', 'Timepoint', 'Date']
df_im_vis   = preprocess.clean_im(df_im, False)
df_cl_vis   = preprocess.clean_cl(df_cl, False)

# Immunological: drop >25% NaN columns, remove confirmed outliers
df_im_mod    = preprocess.remove_nan_cols(df_im_vis, verbose=False)
df_im_mod    = preprocess.remove_outlier_observations(df_im_mod, verbose=False)
df_im_mod    = preprocess.remove_for_modeling(df_im_mod, verbose=False)

# Clinical: drop >25% NaN columns
df_cl_mod    = preprocess.remove_nan_cols(df_cl_vis, verbose=False)
df_cl_mod    = preprocess.remove_for_modeling(df_cl_mod, verbose=False)


df_im_mod : (814, 77)
df_cl_mod : (842, 26)


In [ ]:
# Constructing targets:
print('\nConstructing regression targets from clinical data')
pain_under_load = model.construct_datasets_targets(df_cl_mod, 'pain_under_load', [1, 2])

# Constructing dataset for modeling with targets and im + cl data
print('\nConstructing datasets for modeling:')
df_pain_ul = model.create_model_datasets(df_im_mod, df_cl_mod, pain_under_load, timepoints=[2])


Constructing regression targets from clinical data

  Target distributions:
    pain_reduction                              mean=1.347  std=2.051  [-4.000, 7.100]
    pain_reduction_pct                          mean=19.196  std=32.847  [-100.000, 87.654]

  Target distributions:
    pain_under_load_reduction                   mean=0.551  std=0.753  [-1.000, 3.000]
    pain_under_load_reduction_pct               mean=14.125  std=20.231  [-50.000, 75.000]


## Dataset Overview

In [ ]:
TableReport(df_pain_ul, max_plot_columns=100)



Constructing datasets for modeling:

Total Number of Clinical features: 24
  Dropping 7 Columns before modeling: ['morning_stiffness', 'pain_at_rest', 'pain_daytime', 'pain_night', 'pain_scale', 'pain_under_load', 'response']
  Dropped baseline target cols : ['pain_scale_t1']

Modeling datasets ready: (T1–T2 immunological data + clinical baseline variables:
Shape of Combined Dataset: (116, 95), Number of Patients: 116

Total Number of Clinical features: 24
  Dropping 7 Columns before modeling: ['morning_stiffness', 'pain_at_rest', 'pain_daytime', 'pain_night', 'pain_scale', 'pain_under_load', 'response']
  Dropped baseline target cols : ['pain_under_load_t1']

Modeling datasets ready: (T1–T2 immunological data + clinical baseline variables:
Shape of Combined Dataset: (115, 95), Number of Patients: 115


## SVR Model + MRMR Feature Selection
### + Optuna Hyperparameter Tuning

In [ ]:
pt = PowerTransformer(method='yeo-johnson', standardize=True)

svr_results, svr_feature_freq, svr_per_fold  = model_svr.run_tuned_svr(
        df_pain_ul,
        target_col='pain_under_load_reduction_pct',
        random_state=42,
        target_transformer=pt)

joblib.dump(svr_feature_freq, os.path.join(MODEL_DIR, 'svr_feature_freq.pkl'))
joblib.dump(svr_per_fold, os.path.join(MODEL_DIR, 'svr_per_fold.pkl'))


  CatBoost + Optuna + MRMR — pain_reduction
  n=116, p=92, K=15
  Outer 4×5=20 | Inner 4×5=20 | Model trials=50

─────────────────────────────────────────────────────────────────
  Outer fold 1/20
─────────────────────────────────────────────────────────────────
  MRMR Selected: 15/92 features — ['CD123lo Bas_t2_minus_t1', 'T8lo_t2_minus_t1', 'NK4_t2_minus_t1', 'mDC-1_t2_minus_t1', 'NKT_56+/16+_t2_minus_t1', 'T_CD25+_t2_minus_t1', 'TH_CD25hi_t2_minus_t1', 'Mo_HLADRhi_t2_minus_t1']...
    Trial   1/50: RMSE=1.0052  {'depth': 5, 'learning_rate': 0.23705688269828706, 'l2_leaf_reg': 1.3227908237200787, 'bagging_temperature': 0.31181848267981704}
    Trial   2/50: RMSE=0.9757  {'depth': 8, 'learning_rate': 0.004237260412327668, 'l2_leaf_reg': 1.4941744192199742, 'bagging_temperature': 0.7069343163430255}
    Trial   3/50: RMSE=0.9963  {'depth': 5, 'learning_rate': 0.22379901358345242, 'l2_leaf_reg': 6.843001068876014, 'bagging_temperature': 0.683434887910986}
    Trial   4/50: RMSE=0.9788 

[W 2026-03-19 02:35:42,315] Trial 31 failed with parameters: {'depth': 6, 'learning_rate': 0.007378813252362511, 'l2_leaf_reg': 5.9073372428090885, 'bagging_temperature': 0.8705565266131519} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\mufa\.conda\envs\mt26\lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "p:\UK_Erlangen\Student_folders\Muna Ahmed Farah - IMMO-LDRT01\master-thesis\notebooks\../src\model_catboost_mrmr.py", line 172, in model_objective
    rmses = joblib.Parallel(n_jobs=-1, prefer='threads')(
  File "c:\Users\mufa\.conda\envs\mt26\lib\site-packages\joblib\parallel.py", line 2072, in __call__
    return output if self.return_generator else list(output)
  File "c:\Users\mufa\.conda\envs\mt26\lib\site-packages\joblib\parallel.py", line 1682, in _get_outputs
    yield from self._retrieve()
  File "c:\Users\mufa\.conda\envs\mt26\lib\site-packages\joblib\pa

KeyboardInterrupt: 

### Plot of Feature Frequency List
Top 25 selections shown in plot.

In [ ]:
feature_list = joblib.load(os.path.join(MODEL_DIR, 'svr_feature_freq.pkl'))
model.plot_feature_frequency(feature_list, name='SVR (pain_under_load_reduction_pct)', top=25)

## Jaccard Index of Selected Features
### Pariwise Comparisons for Each Outer Fold (20x20)

In [ ]:
selected_per_fold = joblib.load(os.path.join(MODEL_DIR, 'svr_per_fold.pkl'))
jac_matrix = model.jaccard_scores(selected_per_fold, name='SVR (MRMR Feat. Sel.)')

## SVR on different subsets of selected features

In [ ]:
feature_list = joblib.load(os.path.join(MODEL_DIR, 'svr_feature_freq.pkl'))

svr_sweep_df = model_svr.pls_threshold_analysis(
     df_pain_ul, 
     feature_list, 
     target_col='pain_under_load_reduction_pct',
     random_state=42, 
     target_transformer=pt)
joblib.dump(svr_sweep_df,      os.path.join(MODEL_DIR, 'svr_sweep_df.pkl'))

In [ ]:
# plot sweep
# Plot performances on different feature-tresholds
importlib.reload(model)
model.plot_sweep(svr_sweep_df, name='SVR: Performance Metrics vs. n Features (pain_under_load_reduction_pct)')

# Final SVR Model
## Features Selected in >= /20 outer folds

In [ ]:
feature_list = joblib.load(os.path.join(MODEL_DIR, 'svr_feature_freq.pkl'))
# selecting features in more than 10/20 outer folds:
sel_features = feature_list[feature_list >= 14].index.tolist()

svr_model, svr_X_final, svr_y_pred, svr_patient_err, svr_scaler = model_svr.run_tuned_elasticnet(
    df_pain_ul, 
    sel_features,
    target_col='pain_under_load_reduction_pct', 
    random_state=42,
    target_transformer=pt)

# save results
joblib.dump(svr_model,                os.path.join(MODEL_DIR, 'svr_model.pkl'))
joblib.dump(svr_X_final,              os.path.join(MODEL_DIR, 'svr_X_final.pkl'))
joblib.dump(svr_patient_err,          os.path.join(MODEL_DIR, 'svr_patient_err.pkl'))
joblib.dump(svr_scaler,               os.path.join(MODEL_DIR, 'svr_scaler.pkl'))

## Difficult Patient Predictions

In [ ]:
patient_err = joblib.load(os.path.join(MODEL_DIR, 'svr_patient_err.pkl'))
print(patient_err.to_string())

## SHAP-value Plot

In [ ]:
# Plot Shap values
svr_model= joblib.load(os.path.join(MODEL_DIR, 'svr_model.pkl'))
svr_X_final = joblib.load(os.path.join(MODEL_DIR, 'svr_X_final.pkl'))
svr_scaler = joblib.load(os.path.join(MODEL_DIR, 'svr_scaler.pkl'))

svr_shap = model.plot_shap_elasticnet(svr_model, svr_X_final, svr_scaler)